In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 0.0305,  0.0339, -0.0488, -0.0665,  0.2115, -0.2393, -0.2745, -0.0746,
         -0.2875, -0.3178],
        [ 0.0194,  0.0345, -0.0316, -0.1029,  0.2016, -0.2059, -0.2401, -0.0245,
         -0.4075, -0.3071]], grad_fn=<AddmmBackward0>)

## 5.1.1 自定义块

In [3]:
class MLP(nn.Module):
    # 用模型参数声明层这里，我们声明两个全连接的层
    def __init__(self):
        """
        调用MLP的父类Module的构造函数来执行必要的初始化
        这样，在类实例化时也可以指定其他函数参数，例如模型参数params (稍后将介绍)
        """
        super().__init__()
        self.hidden = nn.Linear(20, 256)        # 隐藏层
        self.out = nn.Linear(256, 10)           # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义
        return self.out(F.relu(self.hidden(X)))

In [4]:
net = MLP()
net(X)

tensor([[-0.1204,  0.1998, -0.0629, -0.1179, -0.3005,  0.1040, -0.0157, -0.2895,
          0.0344, -0.0813],
        [-0.0797,  0.2179, -0.0080, -0.1724, -0.3488,  0.0606,  0.0009, -0.3719,
          0.0474, -0.1315]], grad_fn=<AddmmBackward0>)

## 5.1.2 顺序块

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            """
            这里, module是Module子类的一个实例
            我们把它保存在Module类的成员变量_modules中
            _modules的类型是OrderedDict
            """
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [8]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0654, -0.0266, -0.0726,  0.0099, -0.2449,  0.0303,  0.1183,  0.0063,
          0.3655, -0.2131],
        [-0.0041,  0.0428, -0.0571, -0.0111, -0.1969,  0.0014,  0.1355,  0.0053,
          0.2780, -0.1504]], grad_fn=<AddmmBackward0>)

## 5.1.3 在前向传播函数中执行代码

In [19]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数，因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 服用全连接层这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [20]:
net = FixedHiddenMLP()
net(X)

tensor(0.2490, grad_fn=<SumBackward0>)

In [23]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.2354, grad_fn=<SumBackward0>)